# 17 · Errors

*Beyond the contest* · 6 exercises

Code outside a contest has to fail well: report bad input clearly, give errors a type callers can check, add context without losing the cause, and clean up whatever happens.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ch-17) · [Answer key](https://bloomcode.github.io/cp/python/answer-key.html#ch-17)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    class _Raises:
        def __init__(self, kind):
            self.kind = kind

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            return issubclass(kind, self.kind)

    class pytest:  # noqa: N801
        raises = _Raises

### 17.1 · Parse or report an error  `Core`

**Task.** Write `parse_port(text)` that returns the port number in `text` (spaces around it allowed). Raise `ValueError` with the message `not a number: 'text'` when it is not an integer, and `out of range: N` when it is outside 1 to 65535. Re-raise with `from None` so the caller sees your message, not the parser's.

**Example.** `parse_port(' 8080 ')` -> `8080`; `parse_port('http')` raises `ValueError("not a number: 'http'")`

**Big O.** O(L) for a string of length L.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-17-1) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-17-1)

In [ ]:
# 17.1 Parse or report an error
def parse_port(text):
    raise NotImplementedError

In [ ]:
def test_ex_17_1():
    assert parse_port(" 8080 ") == 8080 and parse_port("1") == 1 and parse_port("65535") == 65535
    with pytest.raises(ValueError, match="not a number: 'http'"):
        parse_port("http")
    for bad, message in (("0", "out of range: 0"), ("70000", "out of range: 70000")):
        with pytest.raises(ValueError, match=message):
            parse_port(bad)


test_ex_17_1()
print('17.1 passed')

### 17.2 · A custom error type  `Core`

**Task.** Create `InsufficientFunds(needed, available)`, an `Exception` subclass that keeps both numbers as attributes and reads `needed N, available M`. Write `withdraw(balance, amount)` that returns the new balance, or raises `InsufficientFunds` when `amount` is more than `balance`. Callers can then catch this one failure and read its fields.

**Example.** `withdraw(100, 30)` -> `70`; `withdraw(20, 50)` raises `InsufficientFunds` with `needed == 50`, `available == 20`

**Big O.** O(1).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-17-2) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-17-2)

In [ ]:
# 17.2 A custom error type
class InsufficientFunds(Exception):
    def __init__(self, needed, available):
        raise NotImplementedError


def withdraw(balance, amount):
    raise NotImplementedError

In [ ]:
def test_ex_17_2():
    assert withdraw(100, 30) == 70 and withdraw(5, 5) == 0
    with pytest.raises(InsufficientFunds) as info:
        withdraw(20, 50)
    assert (info.value.needed, info.value.available) == (50, 20) and str(info.value) == "needed 50, available 20"
    assert isinstance(info.value, Exception)


test_ex_17_2()
print('17.2 passed')

### 17.3 · Wrap an error with context  `Practice`

**Task.** Create `ConfigError(Exception)`. Write `number_setting(settings, key)` that returns `int(settings[key])`, but turns a missing key (`KeyError`) or a bad number (`ValueError`) into `ConfigError("setting 'key'")` raised `from` the original, so the message says where and `__cause__` says why.

**Example.** `number_setting({'port': '80'}, 'port')` -> `80`; `number_setting({}, 'port')` raises `ConfigError` whose `__cause__` is a `KeyError`

**Big O.** O(L) to parse the value.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-17-3) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-17-3)

In [ ]:
# 17.3 Wrap an error with context
class ConfigError:
    pass


def number_setting(settings, key):
    raise NotImplementedError

In [ ]:
def test_ex_17_3():
    assert number_setting({"port": "80"}, "port") == 80
    with pytest.raises(ConfigError) as info:
        number_setting({}, "port")
    assert isinstance(info.value.__cause__, KeyError) and str(info.value) == "setting 'port'"
    with pytest.raises(ConfigError) as info:
        number_setting({"port": "eighty"}, "port")
    assert isinstance(info.value.__cause__, ValueError)


test_ex_17_3()
print('17.3 passed')

### 17.4 · Cleanup that always runs  `Core`

**Task.** Write `run_steps(steps, log)` that appends `"start"` to `log`, calls each function in `steps`, and appends `"done"` however that ends, even when a step raises (the exception still reaches the caller). Use `try` / `finally`.

**Example.** two steps that log `a` and `b` -> `log == ['start', 'a', 'b', 'done']`

**Big O.** O(S) for S steps.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-17-4) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-17-4)

In [ ]:
# 17.4 Cleanup that always runs
def run_steps(steps, log):
    raise NotImplementedError

In [ ]:
def test_ex_17_4():
    log = []
    run_steps([lambda: log.append("a"), lambda: log.append("b")], log)
    assert log == ["start", "a", "b", "done"]

    def fail():
        raise RuntimeError("boom")

    log = []
    with pytest.raises(RuntimeError):
        run_steps([lambda: log.append("a"), fail, lambda: log.append("never")], log)
    assert log == ["start", "a", "done"]


test_ex_17_4()
print('17.4 passed')

### 17.5 · Close resources in reverse order  `Practice`

**Task.** Create `Tracked(name, log)`, a context manager whose `__enter__` appends `open name` and whose `__exit__` appends `close name` (and does not swallow exceptions). Write `use_two(log, fail)` that opens `a` then `b` in one `with` statement, appends `work`, and raises `RuntimeError` when `fail` is true. Both close, in reverse order, either way.

**Example.** `use_two(log, False)` -> `log == ['open a', 'open b', 'work', 'close b', 'close a']`

**Big O.** O(1).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-17-5) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-17-5)

In [ ]:
# 17.5 Close resources in reverse order
class Tracked:
    def __init__(self, name, log):
        raise NotImplementedError

    def __enter__(self):
        raise NotImplementedError

    def __exit__(self, *exc):
        raise NotImplementedError


def use_two(log, fail):
    raise NotImplementedError

In [ ]:
def test_ex_17_5():
    want = ["open a", "open b", "work", "close b", "close a"]
    log = []
    use_two(log, False)
    assert log == want
    log = []
    with pytest.raises(RuntimeError):
        use_two(log, True)
    assert log == want


test_ex_17_5()
print('17.5 passed')

### 17.6 · Collect every error instead of stopping  `Practice`

**Task.** Write `parse_ages(rows)` that parses each row as an age from 0 to 150 and returns `(values, problems)`: the good ages in order, and for each bad row a message `row N: not a number: text` or `row N: out of range: A` (rows count from 1, spaces trimmed). A helper raises `ValueError` for one row; the loop catches it and carries on.

**Example.** `['34', 'x', '200', ' 7 ']` -> `([34, 7], ['row 2: not a number: x', 'row 3: out of range: 200'])`

**Big O.** O(total length of the rows).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-17-6) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-17-6)

In [ ]:
# 17.6 Collect every error instead of stopping
def parse_ages(rows):
    raise NotImplementedError

In [ ]:
def test_ex_17_6():
    assert parse_ages(["34", "x", "200", " 7 "]) == ([34, 7], ["row 2: not a number: x", "row 3: out of range: 200"])
    assert parse_ages([]) == ([], []) and parse_ages(["0", "150"]) == ([0, 150], []) and parse_ages(["-1"]) == ([], ["row 1: out of range: -1"])


test_ex_17_6()
print('17.6 passed')